# Basic TensorFlow neural network

A small binary classification template for MAGIC or MiniBooNE. Run cells from top to bottom.
Edit the settings, layer list, and optimizer to experiment.

Install dependencies from the repository root with `python -m pip install -r methods/requirements.txt`,
then select that Python environment as your notebook kernel.


In [ ]:
from pathlib import Path
import sys

# Allow imports when the notebook starts in methods/ or the repository root.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pipelines" / "datasets.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the ML_sandbox repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay,
)
from pipelines.datasets import load_dataset

print("TensorFlow:", tf.__version__)


## Settings

In [ ]:
DATASET = "magic"  # "magic" or "miniboone"
TEST_SIZE = 0.2
VALIDATION_SIZE = 0.2  # fraction of the non-test data
SEED = 42
EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
THRESHOLD = 0.5

tf.keras.utils.set_random_seed(SEED)


## Load and preprocess

Keep test data for final evaluation. Split validation data before fitting the scaler,
so only the actual training rows determine the scaling.


In [ ]:
X_train, X_test, y_train, y_test, _ = load_dataset(
    DATASET, test_size=TEST_SIZE, preprocessing=None, random_state=SEED,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=VALIDATION_SIZE,
    random_state=SEED, stratify=y_train,
)

scaler = StandardScaler()  # replace with another scaler if desired
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)


## Custom Sequential architecture

Add or remove layers below. Keep one sigmoid output for this binary target.


In [ ]:
tf.keras.backend.clear_session()
model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
], name="basic_classifier")

model.summary()


## Select an optimizer and compile

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
# Alternatives:
# optimizer = tf.keras.optimizers.SGD(learning_rate=LEARNING_RATE, momentum=0.9)
# optimizer = tf.keras.optimizers.RMSprop(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"),
             tf.keras.metrics.AUC(name="auc")],
)


## Train

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True,
    )],
    verbose=1,
)


## Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
epochs = np.arange(1, len(history.history["loss"]) + 1)
for ax, metric in zip(axes, ["loss", "auc"]):
    ax.plot(epochs, history.history[metric], label="Train")
    ax.plot(epochs, history.history[f"val_{metric}"], label="Validation")
    ax.set(xlabel="Epoch", ylabel=metric.upper())
    ax.legend()
fig.tight_layout()
plt.show()


## Evaluate on the held-out test set

Use validation data for architecture, optimizer, and threshold choices. Evaluate the test
set once after those choices are fixed. Keras AUC uses a threshold approximation;
the scikit-learn ROC-AUC below is computed directly from predictions.


In [ ]:
test_metrics = model.evaluate(X_test, y_test, return_dict=True, verbose=0)
print("Keras test metrics:", test_metrics)

y_probability = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
y_prediction = (y_probability >= THRESHOLD).astype(int)
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_probability):.4f}")
print(classification_report(
    y_test, y_prediction, labels=[0, 1],
    target_names=["Background", "Signal"], zero_division=0,
))


## ROC curve and confusion matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_test, y_probability, ax=axes[0])
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_title("Test ROC curve")
ConfusionMatrixDisplay.from_predictions(
    y_test, y_prediction, labels=[0, 1],
    display_labels=["Background", "Signal"], ax=axes[1], colorbar=False,
)
axes[1].set_title(f"Test predictions (threshold={THRESHOLD})")
fig.tight_layout()
plt.show()
